In [34]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
RAW_DIR = ROOT / "data" / "raw"

df_decisions = pl.read_parquet(DATA_DIR / "decisions.parquet")
df_candidates = pl.read_parquet(DATA_DIR / "candidates.parquet")
df_storage = pl.read_parquet(DATA_DIR / "storage_state.parquet")

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [35]:
for name, df in {
    "decisions": df_decisions,
    "candidates": df_candidates,
    "storage_state": df_storage
}.items():
    print(f"\n{name}")
    print(f"Rows: {df.height:,}")
    print(f"Columns: {df.width}")
    print(df.columns)


decisions
Rows: 40
Columns: 10
['decision_id', 'run_id', 'decision_timestamp', 'selected_BoxID', 'candidate_count', 'selected_box_in_candidates', 'valid_for_training', 'selected_BufPos', 'selected_PNo', 'selected_PNoGroup']

candidates
Rows: 729
Columns: 35
['decision_id', 'run_id', 'decision_timestamp', 'BoxID', 'occupied_positions', 'unique_part_count', 'pos_1_PNo', 'pos_1_ONo', 'pos_1_OPos', 'pos_2_PNo', 'pos_2_ONo', 'pos_2_OPos', 'pos_3_PNo', 'pos_3_ONo', 'pos_3_OPos', 'pos_4_PNo', 'pos_4_ONo', 'pos_4_OPos', 'pos_5_PNo', 'pos_5_ONo', 'pos_5_OPos', 'pos_6_PNo', 'pos_6_ONo', 'pos_6_OPos', 'pos_7_PNo', 'pos_7_ONo', 'pos_7_OPos', 'pos_8_PNo', 'pos_8_ONo', 'pos_8_OPos', 'part_numbers', 'order_numbers', 'main_order_numbers', 'selected', 'valid_for_training']

storage_state
Rows: 800
Columns: 6
['decision_id', 'run_id', 'decision_timestamp', 'BufPos', 'BoxID', 'occupied']


In [37]:
valid_ids = df_decisions.filter(
    pl.col("valid_for_training")
).select("decision_id")

check = (
    df_candidates
    .join(valid_ids, on="decision_id", how="inner")
    .group_by("decision_id")
    .agg(
        pl.len().alias("candidate_count"),
        pl.col("selected").sum().alias("selected_count")
    )
)

print("Valid decisions:", check.height)

print(
    "Decisions with exactly one selected candidate:",
    check.filter(pl.col("selected_count") == 1).height
)

print(
    "Decisions with zero or multiple selected candidates:",
    check.filter(pl.col("selected_count") != 1).height
)

check

Valid decisions: 34
Decisions with exactly one selected candidate: 34
Decisions with zero or multiple selected candidates: 0


decision_id,candidate_count,selected_count
str,u32,u32
"""20261016_00000014""",16,1
"""20261016_00000010""",17,1
"""20261016_00000013""",16,1
"""20261016_00000002""",19,1
"""20261016_00000017""",20,1
…,…,…
"""20261016_00000019""",18,1
"""20261016_00000016""",18,1
"""20261016_00000018""",19,1


In [38]:
df_valid = df_candidates.filter(
    pl.col("valid_for_training")
)

print("Shape:", df_valid.shape)

print("\nSchema:")
print(df_valid.schema)

print("\nNull values:")
display(
    df_valid.null_count()
)

print("\nUnique values per column:")
display(
    df_valid.select(
        pl.all().n_unique()
    )
)

Shape: (612, 35)

Schema:
Schema({'decision_id': String, 'run_id': String, 'decision_timestamp': String, 'BoxID': Int64, 'occupied_positions': Int64, 'unique_part_count': Int64, 'pos_1_PNo': Int64, 'pos_1_ONo': Int64, 'pos_1_OPos': Int64, 'pos_2_PNo': Int64, 'pos_2_ONo': Int64, 'pos_2_OPos': Int64, 'pos_3_PNo': Int64, 'pos_3_ONo': Int64, 'pos_3_OPos': Int64, 'pos_4_PNo': Int64, 'pos_4_ONo': Int64, 'pos_4_OPos': Int64, 'pos_5_PNo': Int64, 'pos_5_ONo': Int64, 'pos_5_OPos': Int64, 'pos_6_PNo': Int64, 'pos_6_ONo': Int64, 'pos_6_OPos': Int64, 'pos_7_PNo': Int64, 'pos_7_ONo': Int64, 'pos_7_OPos': Int64, 'pos_8_PNo': Int64, 'pos_8_ONo': Int64, 'pos_8_OPos': Int64, 'part_numbers': String, 'order_numbers': String, 'main_order_numbers': String, 'selected': Boolean, 'valid_for_training': Boolean})

Null values:


decision_id,run_id,decision_timestamp,BoxID,occupied_positions,unique_part_count,pos_1_PNo,pos_1_ONo,pos_1_OPos,pos_2_PNo,pos_2_ONo,pos_2_OPos,pos_3_PNo,pos_3_ONo,pos_3_OPos,pos_4_PNo,pos_4_ONo,pos_4_OPos,pos_5_PNo,pos_5_ONo,pos_5_OPos,pos_6_PNo,pos_6_ONo,pos_6_OPos,pos_7_PNo,pos_7_ONo,pos_7_OPos,pos_8_PNo,pos_8_ONo,pos_8_OPos,part_numbers,order_numbers,main_order_numbers,selected,valid_for_training
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



Unique values per column:


decision_id,run_id,decision_timestamp,BoxID,occupied_positions,unique_part_count,pos_1_PNo,pos_1_ONo,pos_1_OPos,pos_2_PNo,pos_2_ONo,pos_2_OPos,pos_3_PNo,pos_3_ONo,pos_3_OPos,pos_4_PNo,pos_4_ONo,pos_4_OPos,pos_5_PNo,pos_5_ONo,pos_5_OPos,pos_6_PNo,pos_6_ONo,pos_6_OPos,pos_7_PNo,pos_7_ONo,pos_7_OPos,pos_8_PNo,pos_8_ONo,pos_8_OPos,part_numbers,order_numbers,main_order_numbers,selected,valid_for_training
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
34,2,34,21,8,6,6,9,6,5,9,6,6,9,6,7,7,5,7,7,6,8,6,3,6,5,6,6,6,4,26,10,5,2,1


In [41]:
# Columnas de identificación y objetivo
metadata = [
    "decision_id",
    "run_id",
    "decision_timestamp",
    "BoxID",
    "selected",
    "valid_for_training",
]

# Resumen de las variables
feature_columns = [
    c for c in df_valid.columns
    if c not in metadata
]

summary = df_valid.select([
    pl.col(c).n_unique().alias(c)
    for c in feature_columns
])

print("Unique values per feature:")
summary.transpose(
    include_header=True,
    header_name="feature",
    column_names=["unique_values"]
)

Unique values per feature:


feature,unique_values
str,u32
"""occupied_positions""",8
"""unique_part_count""",6
"""pos_1_PNo""",6
"""pos_1_ONo""",9
"""pos_1_OPos""",6
…,…
"""pos_8_ONo""",6
"""pos_8_OPos""",4
"""part_numbers""",26


In [45]:
pno_columns = [
    f"pos_{i}_PNo"
    for i in range(1, 9)
]

for col in pno_columns:
    print(f"\n--- {col} ---")
    print(
        df_valid
        .group_by(col)
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
    )



--- pos_1_PNo ---
shape: (6, 2)
┌───────────┬───────┐
│ pos_1_PNo ┆ count │
│ ---       ┆ ---   │
│ i64       ┆ u32   │
╞═══════════╪═══════╡
│ 0         ┆ 488   │
│ 832       ┆ 68    │
│ 811       ┆ 33    │
│ 831       ┆ 12    │
│ 813       ┆ 7     │
│ 800       ┆ 4     │
└───────────┴───────┘

--- pos_2_PNo ---
shape: (5, 2)
┌───────────┬───────┐
│ pos_2_PNo ┆ count │
│ ---       ┆ ---   │
│ i64       ┆ u32   │
╞═══════════╪═══════╡
│ 0         ┆ 511   │
│ 814       ┆ 44    │
│ 832       ┆ 31    │
│ 813       ┆ 23    │
│ 800       ┆ 3     │
└───────────┴───────┘

--- pos_3_PNo ---
shape: (6, 2)
┌───────────┬───────┐
│ pos_3_PNo ┆ count │
│ ---       ┆ ---   │
│ i64       ┆ u32   │
╞═══════════╪═══════╡
│ 0         ┆ 530   │
│ 812       ┆ 34    │
│ 800       ┆ 17    │
│ 813       ┆ 17    │
│ 811       ┆ 12    │
│ 814       ┆ 2     │
└───────────┴───────┘

--- pos_4_PNo ---
shape: (7, 2)
┌───────────┬───────┐
│ pos_4_PNo ┆ count │
│ ---       ┆ ---   │
│ i64       ┆ u32   │
╞═════════

In [44]:
print("Valores de occupied_positions:")
df_valid["occupied_positions"].value_counts().sort("occupied_positions")

print("\nValores de unique_part_count:")
df_valid["unique_part_count"].value_counts().sort("unique_part_count")

Valores de occupied_positions:

Valores de unique_part_count:


unique_part_count,count
i64,u32
0,352
1,57
2,54
3,58
4,78
5,13


In [ ]:
dfs_buffer = []

for run_id in ["20261016", "20261020"]:

    folder = RAW_DIR / run_id / "tblbufferpos"

    files = list(folder.rglob("*.parquet"))

    print(f"{run_id}: {len(files)} archivos")

    for file in files:
        df = pl.read_parquet(file).with_columns(
            pl.lit(run_id).alias("run_id")
        )
        dfs_buffer.append(df)

df_bufferpos = pl.concat(dfs_buffer, how="diagonal_relaxed")

print("\nTotal rows:", df_bufferpos.height)
print("Columns:", df_bufferpos.columns)

In [ ]:
df_decisions.head(10)

In [ ]:
# decisions per run
(
    df_decisions
    .group_by("run_id")
    .agg(
        pl.len().alias("num_decisions"),
        pl.col("candidate_count").mean().alias("avg_candidates"),
        pl.col("candidate_count").min().alias("min_candidates"),
        pl.col("candidate_count").max().alias("max_candidates"),
        (~pl.col("selected_box_in_candidates"))
            .sum()
            .alias("missing_selected_boxes")
    )
    .sort("run_id")
)

In [ ]:
# distribution candidaate
(
    df_decisions
    .group_by(["run_id", "candidate_count"])
    .len()
    .sort(["run_id", "candidate_count"])
)

In [ ]:
# missing selections
df_missing = (
    df_decisions
    .filter(~pl.col("selected_box_in_candidates"))
    .sort(["run_id", "decision_timestamp"])
)

df_missing

In [ ]:
# details missing selections
df_missing.select(
    "decision_id",
    "run_id",
    "decision_timestamp",
    "selected_BoxID",
    "selected_BufPos",
    "selected_PNo",
    "candidate_count"
)

In [ ]:
# selected box in snapshot
(
    df_missing
    .join(
        df_storage,
        on=["decision_id", "run_id", "decision_timestamp"],
        how="left"
    )
    .group_by(["decision_id", "selected_BoxID"])
    .agg(
        pl.col("BoxID").unique().alias("boxes_in_storage"),
        pl.col("BufPos").sort().alias("storage_positions")
    )
)

In [ ]:
# decision candidaates
example_decision = df_decisions["decision_id"][0]

(
    df_candidates
    .filter(pl.col("decision_id") == example_decision)
    .sort("BoxID")
)

In [ ]:
from datetime import datetime, timezone

start_time = datetime(2026, 10, 16, 13, 28, 0, tzinfo=timezone.utc)
end_time = datetime(2026, 10, 16, 13, 32, 0, tzinfo=timezone.utc)

df_events_around = (
    df_bufferpos
    .filter(
        (pl.col("run_id") == "20261016") &
        (pl.col("timestamp") >= start_time) &
        (pl.col("timestamp") <= end_time)
    )
    .sort("timestamp")
    .select(
        "timestamp",
        "action",
        "ResourceId",
        "BufNo",
        "BufPos",
        "BoxID",
        "PNo",
        "ONo",
        "OPos"
    )
)
df_events_around

In [ ]:
# check selections from candidates
(
    df_candidates
    .group_by("decision_id")
    .agg(
        pl.len().alias("num_candidates"),
        pl.col("selected").sum().alias("num_selected"),
        pl.col("BoxID").n_unique().alias("unique_boxes")
    )
    .sort("decision_id")
)

In [ ]:
# distribution selected boxes
(
    df_candidates
    .filter(pl.col("selected"))
    .group_by(["run_id", "BoxID"])
    .len()
    .sort(["run_id", "len"], descending=[False, True])
)

In [ ]:
# storage state
example_decision = df_decisions["decision_id"][0]

(
    df_storage
    .filter(pl.col("decision_id") == example_decision)
    .sort("BufPos")
)

In [ ]:
# occupied positions
(
    df_storage
    .group_by(["decision_id", "run_id", "decision_timestamp"])
    .agg(
        pl.col("occupied").sum().alias("occupied_positions"),
        pl.col("BoxID").filter(pl.col("occupied")).n_unique()
            .alias("unique_boxes")
    )
    .sort(["run_id", "decision_timestamp"])
)

In [ ]:
# ¿Hay más de una fila para la misma posición dentro de un snapshot?
duplicate_positions = (
    df_storage
    .group_by(["decision_id", "BufPos"])
    .len()
    .filter(pl.col("len") > 1)
)

print("Duplicate storage positions:")
print(duplicate_positions)

# ¿Hay cajas duplicadas en distintas posiciones?
duplicate_boxes = (
    df_storage
    .filter(pl.col("occupied"))
    .group_by(["decision_id", "BoxID"])
    .len()
    .filter(pl.col("len") > 1)
)

print("\nBoxes occupying multiple positions:")
print(duplicate_boxes)

In [ ]:
df_candidates.select(
    "BoxID",
    "occupied_positions",
    "unique_part_count",
    "part_numbers",
    "order_numbers",
    "main_order_numbers"
).head(20)

In [ ]:
box_example = df_candidates["BoxID"][0]

position_columns = [
    col for col in df_candidates.columns
    if col.startswith("pos_")
]

(
    df_candidates
    .filter(pl.col("BoxID") == box_example)
    .select(
        "decision_id",
        "decision_timestamp",
        "BoxID",
        *position_columns,
        "selected"
    )
)

In [ ]:
run_id = df_missing["run_id"][0]

buffer_files = sorted(
    (RAW_DIR / run_id / "tblbufferpos").rglob("*.parquet")
)

df_buffer_raw = pl.concat(
    [pl.read_parquet(file) for file in buffer_files],
    how="diagonal_relaxed"
)

df_buffer_raw.head()

In [ ]:
problem = df_missing.row(0, named=True)

selected_timestamp = problem["decision_timestamp"]
selected_box = problem["selected_BoxID"]

print("Run:", problem["run_id"])
print("Timestamp:", selected_timestamp)
print("Selected box:", selected_box)

(
    df_buffer_raw
    .filter(
        (pl.col("ResourceId").cast(pl.Utf8) == "17")
        & (pl.col("timestamp").cast(pl.Utf8) >= selected_timestamp[:19])
    )
    .sort("timestamp")
    .select(
        "timestamp",
        "action",
        "ResourceId",
        "BufNo",
        "BufPos",
        "PNo",
        "BoxID",
        "PNoGroup"
    )
    .head(40)
)

In [ ]:
# storage events
(
    df_bufferpos
    .filter(
        (pl.col("ResourceId") == 17) &
        (pl.col("BufNo") == 1)
    )
    .group_by(["run_id", "action"])
    .agg(
        pl.len().alias("num_events"),
        pl.col("BufPos").n_unique().alias("unique_positions"),
        pl.col("BoxID").filter(pl.col("BoxID") > 0).n_unique().alias("boxes")
    )
    .sort(["run_id", "action"])
)

In [ ]:
# first events
df_bufferpos.filter(
        (pl.col("run_id") == "20261016") &
        (pl.col("ResourceId") == 17) &
        (pl.col("BufNo") == 1) &
        (pl.col("action") == "update")
    ).sort("timestamp").select("timestamp", "BufPos", "BoxID", "PNo", "ONo", "OPos").head(30)

In [ ]:
# updates storage
storage_updates = (
    df_bufferpos
    .filter(
        (pl.col("ResourceId") == 17) &
        (pl.col("BufNo") == 1) &
        (pl.col("action") == "update")
    )
    .select(
        "run_id",
        pl.col("timestamp").alias("event_timestamp"),
        "BufPos",
        "BoxID"
    )
)

# Eventos de selección
selection_events = (
    df_bufferpos
    .filter(
        (pl.col("ResourceId") == 17) &
        (pl.col("BufNo") == 3) &
        (pl.col("action") == "update") &
        (pl.col("BoxID") > 0)
    )
    .select(
        "run_id",
        pl.col("timestamp").alias("event_timestamp"),
        pl.col("BoxID").alias("selected_BoxID")
    )
)

# Buscar timestamps idénticos
same_timestamp = (
    storage_updates
    .join(
        selection_events,
        on=["run_id", "event_timestamp"],
        how="inner"
    )
    .sort(["run_id", "event_timestamp"])
)

print("Storage updates:", storage_updates.height)
print("Selections:", selection_events.height)
print("Same timestamp:", same_timestamp.height)

same_timestamp

In [1]:
import polars as pl
from pathlib import Path

ROOT = Path.cwd()

decisions = pl.read_parquet(
    ROOT / "data/processed/decisions.parquet"
)

buffer = pl.read_parquet(
    ROOT / "data/interim/combined/tblbufferpos.parquet"
)

invalid = decisions.filter(
    ~pl.col("valid_for_training")
)

for decision in invalid.to_dicts():

    run_id = decision["run_id"]
    box_id = decision["selected_BoxID"]
    decision_ts = decision["decision_timestamp"]

    print("\n" + "=" * 90)
    print(f"Decision: {decision['decision_id']}")
    print(f"Selected BoxID: {box_id}")
    print(f"Selected BoxPos: {decision['selected_BoxPos']}")
    print(f"Decision timestamp: {decision_ts}")
    print("=" * 90)

    events = (
        buffer
        .filter(
            (pl.col("run_id") == run_id)
            & (pl.col("ResourceId") == 17)
            & (pl.col("BoxID") == box_id)
            & (pl.col("BufNo").is_in([1, 3]))
            & (pl.col("PNo") == 29)
        )
        .with_columns(
            pl.col("timestamp").cast(pl.String)
        )
        .sort("timestamp")
        .select([
            "timestamp",
            "action",
            "BufNo",
            "BufPos",
            "BoxID",
            "PNo",
        ])
    )

    print(events)

FileNotFoundError: El sistema no puede encontrar la ruta especificada. (os error 3): ...sers/Gaby/PycharmProjects/resource_control_ml/notebooks/data/processed/decisions.parquet (set POLARS_VERBOSE=1 to see full path)